# 12F-06 — WLTP/JRC BEV EDA

**Question:** How does the electric JRC population behave, and which values are directly comparable?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
JRC_FILE = REPO_ROOT / "etl/data/raw/wltp_jrc/Data_PV_fleet_2021_EU_PYCSIS.xlsx"
jrc = pd.read_excel(JRC_FILE, sheet_name="Sheet1", engine="openpyxl")
assert pd.api.types.is_bool_dtype(jrc["is_electric"]), jrc["is_electric"].dtype
bev = jrc[jrc["is_electric"].fillna(False).astype(bool)].copy()
energy_field = "Declared electric consumption value (OEM) [Wh/km]"
bev[energy_field] = pd.to_numeric(bev[energy_field], errors="coerce")
print(f"{len(bev)} electric rows; {bev[energy_field].notna().sum()} declared Wh/km values")


## Explicit source-unit distribution

In [ ]:
bev[[energy_field, "Electric range (OEM) [km]", "Vehicle mass (WLTP) [kg]"]].describe().T


## Declared versus simulated context

In [ ]:
simulated = "Declared electric consumption value (Simulated) [Wh/km]"
comparison = bev[["OEM anon", "Model anon", energy_field, simulated]].dropna().copy()
comparison["simulated_minus_oem_wh_km"] = comparison[simulated] - comparison[energy_field]
comparison.head(10)


## Does consumption move with mass?

In [ ]:
import matplotlib.pyplot as plt

plot_data = bev.assign(mass_kg=pd.to_numeric(bev["Vehicle mass (WLTP) [kg]"], errors="coerce")).dropna(subset=["mass_kg", energy_field])
ax = plot_data.plot.scatter(x="mass_kg", y=energy_field, alpha=0.65, figsize=(7, 4))
ax.set(title="JRC electric source records")
plt.show()


## What did we learn?

The OEM field is directly labelled Wh/km. Declared, simulated, and real-world fields remain different result contexts and must not be silently averaged.